# Solutions to L15

```{solution-start} l15-membrane-current
:class: dropdown
:label: sol-l15-membrane-current
See solution below. 
```

## Eliminating membrane current

At time step $t_n$, we approximate the time derivative using a forward finite difference scheme:
$$
        \frac{v^n-v^{n-1}}{\Delta t} = \frac{1}{C_m} (I_m^n - I_{\text{ion}}(v^{n-1}))
$$
Here, the unknown membrane current $I_m^n$ is treated implicitly, whereas the nonlinear ionic current is treated explicitly and evaluated at the previous, known time step $v^{n-1}$. 

First, we rearrange the equation to isolate the unknown membrane current $I_m^n$:
$$
        I_m^n = \frac{C_m}{\Delta t} (v^n - v^{n-1}) + I_{\text{ion}}(v^{n-1})
$$

Next, we substitute the definition of the unknown transmembrane potential, $v^n = u_i^n - u_e^n$. (We leave $v^{n-1}$ as is, since it is already a known quantity computed in the previous step):
$$
        I_m^n = \frac{C_m}{\Delta t} (u_i^n - u_e^n - v^{n-1}) + I_{\text{ion}}(v^{n-1})
$$

Finally, to simplify the implementation of our weak form, we factor out $\frac{C_m}{\Delta t}$ and group all the known terms from the previous time step into a single explicit source variable, $f$:
$$
        I_m^n = \frac{C_m}{\Delta t} \left[ u_i^n - u_e^n - \underbrace{\left( v^{n-1} - \frac{\Delta t}{C_m} I_{\text{ion}}(v^{n-1}) \right)}_{f} \right]
$$

Thus, our membrane current is simply $I_m^n = \frac{C_m}{\Delta t} (u_i^n - u_e^n - f)$.

```{solution-end}
```

```{solution-start} l15-membrane-stimulus
:class: dropdown
:label: sol-l15-membrane-stimulus
See solution below. 
```

## Membrane stimulus

Add $I_{\rm stim}$ to the RHS as 
```python
L = (Cm / dt * v_n - I_ion - I_stim) * (tr_vi - tr_ve) * dGamma
```

Inside the simulation loop, use 
```python
if 0.1 <= t <= 1.0:
    I_stim.value = -120.0
else:
    I_stim.value = 0.0
```

```{solution-end}
```

```{solution-start} l15-conduction-velocity
:class: dropdown
:label: sol-l15-conduction-velocity
See solution below. 
```

## Computing conduction velocity

```python
if t1_val > 0 and t2_val > 0:
    cv = ((eval_pt_2[0] - eval_pt_1[0]) / (t2_val - t1_val)) * 1000.0
    print(
        f"Activation at C{cv_cell_1}: {t1_val:.3f} ms | C{cv_cell_2}: {t2_val:.3f} ms"
    )
    print(f"Calculated Macroscopic CV: {cv:.2f} cm/s.")
else:
    print("Warning: Wave did not reach both cells.")
```

```{solution-end}
```

```{solution-start} l15-gna-nonuniform
:class: dropdown
:label: sol-l15-gna-nonuniform
See solution below. 
```

## Non-uniform gNa

```python
area_body = area_tot - area_clusters

channels_total = g_Na_const.value * area_tot
channels_baseline = g_Na_baseline * area_body
channels_left_for_tips = channels_total - channels_baseline

peak_gNa = channels_left_for_tips / area_clusters
print(f"Computed global cluster peak gNa: {peak_gNa:.2f} mS/µF")

# Scale all functions: map the [0, 1] mask to [baseline, peak]
for g_k in g_Na_dict.values():
    
    # Step A: How much extra conductance goes into the tips?
    extra_gNa = peak_gNa - g_Na_baseline
    
    # Step B: Scale the [0, 1] mask to this new maximum difference
    g_k.x.array[:] = g_k.x.array[:] * extra_gNa
    
    # Step C: Shift the entire cell up by the baseline amount
    g_k.x.array[:] = g_k.x.array[:] + g_Na_baseline
```

```{solution-end}
```